# Can we predict a warranty claim within a year?

Target: `claim_within_year`. Plan: inspect data quality, compare simple and tree-based models with cross-validation against a no-skill baseline, then confirm on a held-out test set.

`device_serial` is an identifier and is excluded from the features.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv')
df.head()

## 1. Data quality

In [ ]:
print(df.shape); print(df.isna().sum().to_string())
print('duplicate serials:', df.device_serial.duplicated().sum())
print('claim rate:', df.claim_within_year.mean().round(4))
df.describe().T

In [ ]:
print('daily_hours > 24:', (df.daily_hours>24).sum(), '| ==0:', (df.daily_hours==0).sum())
print('device_age_years < 0.05:', (df.device_age_years<0.05).sum())
print('price_eur <= 0:', (df.price_eur<=0).sum())
fig,ax=plt.subplots(1,3,figsize=(13,3))
for a,c in zip(ax,['device_age_years','price_eur','daily_hours']): df[c].hist(bins=50,ax=a); a.set_title(c)
plt.tight_layout()

`daily_hours` values above 24 are physically impossible for a *daily average*, so they are probably entry errors (the column is self-reported at registration). I keep the rows and test sensitivity to this later by clipping at 24.

## 2. Relationship of each feature with the target

In [ ]:
print(df.groupby('channel').claim_within_year.agg(['mean','size']).round(3))
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,c in zip(ax,['device_age_years','price_eur','daily_hours']):
    g=df.groupby(pd.qcut(df[c],10,duplicates='drop'),observed=True).claim_within_year.mean()
    a.plot(range(len(g)),g.values,'o-'); a.set_title(f'claim rate by {c} decile'); a.set_ylim(0,None); a.axhline(df.claim_within_year.mean(),ls='--',c='gray')
plt.tight_layout()

## 3. Models

Split 80/20 stratified; the test set is only touched once at the end. Metrics: ROC AUC and average precision (PR AUC), since only ~13% of devices have a claim and accuracy would be misleading.

In [ ]:
X=df.drop(columns=['device_serial','claim_within_year']); y=df.claim_within_year
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),
 'grad. boosting':Pipeline([('p',ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)])),
                           ('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    auc=cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc'); ap=cross_val_score(m,Xtr,ytr,cv=cv,scoring='average_precision')
    rows.append((n,auc.mean(),auc.std(),ap.mean(),ap.std()))
pd.DataFrame(rows,columns=['model','AUC','AUC sd','AP','AP sd']).round(3)

## 4. Held-out test performance

In [ ]:
res={}
for n in ['logistic','grad. boosting']:
    models[n].fit(Xtr,ytr); res[n]=models[n].predict_proba(Xte)[:,1]
    print(f'{n}: test AUC={roc_auc_score(yte,res[n]):.3f}  AP={average_precision_score(yte,res[n]):.3f}  (base rate {yte.mean():.3f})')
rng=np.random.default_rng(SEED); p=res['logistic']; yy=yte.values
b=[roc_auc_score(yy[i],p[i]) for i in (rng.integers(0,len(yy),len(yy)) for _ in range(500))]
print('logistic AUC 95% bootstrap CI:',np.percentile(b,[2.5,97.5]).round(3))
fig,ax=plt.subplots(1,2,figsize=(10,4))
for n,p in res.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_title('ROC'); ax[0].legend()
ax[1].axhline(yte.mean(),c='k',ls='--'); ax[1].set_title('Precision-recall'); ax[1].set_xlabel('recall')
plt.tight_layout()

## 5. What drives the prediction, and is it usable?

In [ ]:
m=models['logistic']; names=m['p'].get_feature_names_out()
print(pd.Series(m['m'].coef_[0],index=names).round(3).sort_values().to_string())
pi=permutation_importance(models['grad. boosting'],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
print('\npermutation importance (AUC drop), gradient boosting')
print(pd.Series(pi.importances_mean,index=X.columns).round(3).sort_values(ascending=False).to_string())
q=pd.qcut(res['logistic'],5,labels=False)
print('\nobserved claim rate by predicted-risk quintile (test):'); print(yte.groupby(q).mean().round(3).to_string())

## 6. Robustness: clip `daily_hours` at 24

In [ ]:
Xc=X.copy(); Xc['daily_hours']=Xc.daily_hours.clip(upper=24)
a,_,c_,_=train_test_split(Xc,y,test_size=0.2,stratify=y,random_state=SEED)
print('logistic CV AUC, clipped:',cross_val_score(models['logistic'],a,c_,cv=cv,scoring='roc_auc').mean().round(3))

## Conclusions

(Numbers from the run with seed 0; re-running reproduces them.)

- **Yes, moderately.** 5-fold CV AUC is ~0.72 for logistic regression (vs 0.50 baseline) and test AUC is 0.76 (bootstrap 95% CI ~0.72-0.80). Average precision 0.34 vs a 0.126 base rate.
- **Useful for ranking, not for confident individual calls.** The top predicted-risk quintile has a ~32% claim rate against ~3% in the bottom quintile, a ~10x spread. A hard yes/no classifier would have low precision.
- **Drivers:** older device at sale, higher daily use, higher price, and the `partner` channel all raise risk. Device age is the strongest.
- **Gradient boosting did not beat logistic regression**, which suggests the signal is mostly simple and monotone, with little to gain from nonlinear modelling on these four features.
- **Data caveats:** 23 rows have `daily_hours` > 24 (impossible); clipping them does not change results. Only ~880 positives, so estimates are noisy. Claim timing and cost are not in the data.